# A.X-Encoder 평가 v2 — 학습 실패 수정판

## 1차 실행에서 무슨 일이 있었나

| 지표 | 1차 결과 | 정상이라면 |
|---|---|---|
| A.X-Encoder (normal) | **0.680** | ≥ 0.94 |
| TF-IDF response-only | 0.940 | — |
| empty(문맥 제거) AUC | **0.822** | normal보다 낮아야 |
| normal AUC | 0.739 | empty보다 높아야 |

**149M 파라미터 사전학습 인코더가 문자 n-gram 로지스틱 회귀에 26%p 졌다.**
게다가 history를 **없앴을 때 오히려 성능이 올랐다**(0.700 > 0.680, AUC 0.822 > 0.739).

이건 "문맥을 안 본다"가 아니라 **"아무것도 못 배웠다"** 의 증상이다.
누출된 데이터에서 정답은 응답 표면에 그대로 놓여 있어서 TF-IDF도 0.94를 맞힌다.
그걸 못 맞혔다는 건 모델 탓도 데이터 탓도 아니고 **학습 설정 탓**이다.

결정적 증거는 확률 분포다. 오분류 19건이 **전부 0.423~0.646** 안에 있다.
threshold를 0.25로 낮추면 100%가 부적절, 0.65로 올리면 5%만 부적절 —
확률이 0.5 언저리 좁은 띠에 뭉쳐 있다. **모델이 초기값에서 거의 안 움직였다.**

## 원인과 수정

| 항목 | 1차 | v2 | 이유 |
|---|---|---|---|
| `LR` | 2e-5 | **5e-5** | ModernBERT는 BERT보다 높은 학습률이 필요하다. 2e-5는 BERT 기준값 |
| `EPOCHS` | 6 | **15** | 80행 ÷ batch 8 = epoch당 10스텝. 6 epoch = 총 60스텝뿐이었다 |

> 제 1차 노트북의 §5 판정 로직에 결함이 있었습니다. **"모델이 학습됐는가"를
> 확인하지 않고 곧바로 "문맥 미사용"을 출력**했습니다. v2는 그 게이트를 넣었습니다.

## v2에서 추가된 것

- **§4 학습 점검** — 단일 fold 30초. 여기서 걸러내면 200초를 낭비하지 않는다
- **학습셋 자체 정확도** — 학습 데이터도 못 맞히면 그건 100% 학습 실패다
- **확률 분포 진단** — 표준편차와 구간 분포로 "결정을 내렸는가"를 본다
- **§6 판정 게이트** — 학습이 실패했으면 절제 실험 해석을 거부한다

## 이 데이터의 역설적 쓸모

누출된 100행은 **학습 파이프라인 점검용으로는 최고의 도구**다.
정답이 표면에 있으니 제대로 학습되면 반드시 0.94 근처가 나와야 한다.
**못 넘으면 파이프라인이 고장난 것이다.** v2는 이걸 합격 기준으로 쓴다.


## 1. 설치 · 데이터 로드

In [ ]:
!pip -q install -U "transformers>=4.48" "scikit-learn>=1.3"
import transformers, torch
print("transformers", transformers.__version__, "| torch", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "없음 (런타임을 GPU로 바꿀 것)")

In [ ]:
CSV_PATH = "dialogue_context_appropriateness_100.csv"   # 다른 파일로 평가하려면 여기만 바꾼다

import os, numpy as np, pandas as pd
if not os.path.exists(CSV_PATH):
    from google.colab import files
    CSV_PATH = list(files.upload().keys())[0]

df = pd.read_csv(CSV_PATH)
need = {"pair_id", "history", "response", "label"}
assert not (need - set(df.columns)), f"컬럼 누락: {need - set(df.columns)}"
df = df.dropna(subset=["history", "response", "label"]).reset_index(drop=True)
df["history"] = df["history"].astype(str)
df["response"] = df["response"].astype(str)

y = (df["label"].astype(str).str.strip() == "부적절").astype(int).values
groups = df["pair_id"].astype(str).values

print(f"{CSV_PATH} | {len(df)}행 | 부적절 {y.sum()} / 적절 {(1-y).sum()} | pair {len(set(groups))}개")

## 2. 누출 기준선 — 이번에는 **합격 기준**으로도 쓴다

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import StratifiedGroupKFold, cross_val_predict
from sklearn.metrics import accuracy_score

def surface_baseline(texts, name):
    pipe = make_pipeline(TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4), min_df=1),
                         LogisticRegression(max_iter=2000))
    pred = cross_val_predict(pipe, np.array(texts), y,
                             cv=StratifiedGroupKFold(5, shuffle=True, random_state=42), groups=groups)
    acc = accuracy_score(y, pred)
    print(f"{name:22s} acc = {acc:.3f}")
    return acc

BASE_RESP = surface_baseline(df["response"], "response-only")
BASE_HIST = surface_baseline(df["history"],  "history-only")
both_sides = (df.groupby("response")["label"].nunique() > 1).sum()
print(f"\n양쪽 라벨에 등장하는 response: {both_sides} / {df['response'].nunique()}")

# 학습 파이프라인 합격선: 누출된 데이터라면 TF-IDF 근처까지 가야 정상이다.
PASS_BAR = max(0.70, BASE_RESP - 0.05)
print(f"\n>>> 이번 실행의 학습 합격선: OOF 정확도 {PASS_BAR:.3f} 이상")
print("    (누출 데이터에서 TF-IDF도 맞히는 걸 못 맞히면 학습이 고장난 것)")

## 3. 모델 · 하이퍼파라미터

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL_NAME = "skt/A.X-Encoder-base"
MAX_LEN = 256
EPOCHS  = 15      # 1차 6 → 15. 80행/batch8 = epoch당 10스텝이라 6 epoch은 60스텝뿐이었다
BATCH   = 8
LR      = 5e-5    # 1차 2e-5 → 5e-5. ModernBERT는 BERT 기준값(2e-5)보다 높은 LR이 필요하다

device   = torch.device("cuda" if torch.cuda.is_available() else "cpu")
cap      = torch.cuda.get_device_capability(0) if torch.cuda.is_available() else (0, 0)
USE_BF16 = cap[0] >= 8
print(f"device={device} | bf16={USE_BF16} | EPOCHS={EPOCHS} | LR={LR}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def load_model():
    try:
        m = AutoModelForSequenceClassification.from_pretrained(
            MODEL_NAME, num_labels=2, attn_implementation="sdpa")
    except Exception:
        m = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
    return m.to(device)

def encode(h, r):
    return tokenizer(list(h), list(r), truncation="only_first",
                     max_length=MAX_LEN, padding=True, return_tensors="pt")

In [ ]:
from torch.utils.data import TensorDataset, DataLoader
from transformers import get_linear_schedule_with_warmup

def train_one_fold(tr_idx, va_idx, seed, verbose=False):
    torch.manual_seed(seed); np.random.seed(seed)
    model = load_model()

    enc = encode(df["history"].values[tr_idx], df["response"].values[tr_idx])
    ds = TensorDataset(enc["input_ids"], enc["attention_mask"],
                       torch.tensor(y[tr_idx], dtype=torch.long))
    dl = DataLoader(ds, batch_size=BATCH, shuffle=True)

    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
    total = len(dl) * EPOCHS
    sch = get_linear_schedule_with_warmup(opt, max(1, int(total * 0.1)), total)
    amp = torch.bfloat16 if USE_BF16 else torch.float32

    losses = []
    model.train()
    for ep in range(EPOCHS):
        ep_loss = 0.0
        for ids, mask, lab in dl:
            ids, mask, lab = ids.to(device), mask.to(device), lab.to(device)
            with torch.autocast("cuda", dtype=amp, enabled=USE_BF16):
                loss = model(input_ids=ids, attention_mask=mask, labels=lab).loss
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step(); sch.step(); opt.zero_grad()
            ep_loss += loss.item()
        losses.append(ep_loss / len(dl))
        if verbose and (ep + 1) % 3 == 0:
            print(f"    epoch {ep+1:2d}/{EPOCHS}  loss {losses[-1]:.4f}")

    @torch.no_grad()
    def predict(hists, resps):
        model.eval()
        out = []
        for i in range(0, len(resps), 32):
            e = encode(hists[i:i+32], resps[i:i+32])
            with torch.autocast("cuda", dtype=amp, enabled=USE_BF16):
                lg = model(input_ids=e["input_ids"].to(device),
                           attention_mask=e["attention_mask"].to(device)).logits
            out.append(torch.softmax(lg.float(), -1)[:, 1].cpu().numpy())
        return np.concatenate(out)

    # 학습셋 자체 정확도 — 이게 낮으면 일반화 이전에 학습이 안 된 것이다
    p_tr = predict(df["history"].values[tr_idx], df["response"].values[tr_idx])
    train_acc = (((p_tr > 0.5).astype(int)) == y[tr_idx]).mean()

    h_va, r_va = df["history"].values[va_idx], df["response"].values[va_idx]
    res = {
        "normal":  predict(h_va, r_va),
        "swapped": predict(np.roll(h_va, 1), r_va),
        "empty":   predict(np.array([""] * len(r_va)), r_va),
        "train_acc": train_acc,
        "losses": losses,
    }
    del model; torch.cuda.empty_cache()
    return res

## 4. 학습 점검 — 단일 fold, 약 30초

**전체 CV(200초+)를 돌리기 전에 여기서 먼저 확인한다.** 세 가지를 본다.

| 확인 | 정상 | 실패 시 의미 |
|---|---|---|
| loss가 떨어지는가 | 0.69 → 0.1 이하 | 안 떨어지면 학습률·스텝 문제 |
| **학습셋 정확도** | ≥ 0.95 | 낮으면 일반화 이전에 **암기조차 실패** |
| 확률 분포 | 0/1 양극으로 벌어짐 | 0.5에 몰리면 결정을 못 내린 것 |

1차 실행은 세 번째에서 걸렸다 — 확률이 전부 0.42~0.65 안에 있었다.


In [ ]:
import time
from sklearn.model_selection import StratifiedGroupKFold

cv0 = StratifiedGroupKFold(5, shuffle=True, random_state=42)
tr0, va0 = next(iter(cv0.split(df, y, groups)))

print(f"학습 {len(tr0)}행 / 검증 {len(va0)}행 · {EPOCHS} epoch · LR {LR}")
t0 = time.time()
chk = train_one_fold(tr0, va0, seed=42, verbose=True)
print(f"({time.time()-t0:.0f}초)\n")

va_acc = (((chk["normal"] > 0.5).astype(int)) == y[va0]).mean()
p = chk["normal"]
decisive = ((p < 0.2) | (p > 0.8)).mean()

print(f"loss        : {chk['losses'][0]:.4f} → {chk['losses'][-1]:.4f}")
print(f"학습셋 정확도 : {chk['train_acc']:.3f}   (목표 ≥ 0.95)")
print(f"검증 정확도   : {va_acc:.3f}")
print(f"확률 표준편차 : {p.std():.3f}   범위 {p.min():.3f}~{p.max():.3f}")
print(f"확신 비율     : {decisive:.0%}  (0.2 미만 또는 0.8 초과)")

print("\n" + "=" * 60)
if chk["train_acc"] < 0.90:
    print("[학습 실패] 학습 데이터조차 못 맞힌다.")
    print(f"  → EPOCHS를 {EPOCHS*2}, LR을 8e-5 로 올려 §3부터 다시 실행할 것.")
    print("  → 여기서 통과 못 하면 아래 전체 CV는 돌릴 의미가 없다.")
elif decisive < 0.5:
    print("[결정 미흡] 확률이 0.5 근처에 몰려 있다. epoch을 더 늘릴 것.")
else:
    print("[통과] 학습이 정상 동작한다. 아래 전체 CV로 진행.")
print("=" * 60)

## 5. 전체 교차검증 — §4를 통과한 뒤에만 실행

5-fold × 3 seed = 15회. epoch을 늘렸으므로 **6~8분** 예상.


In [ ]:
SEEDS = [42, 7, 2026]
CONDS = ["normal", "swapped", "empty"]
oof = {s: {c: np.zeros(len(df)) for c in CONDS} for s in SEEDS}
train_accs = []

t0 = time.time()
for seed in SEEDS:
    cv = StratifiedGroupKFold(5, shuffle=True, random_state=seed)
    for k, (tr, va) in enumerate(cv.split(df, y, groups)):
        r = train_one_fold(tr, va, seed)
        for c in CONDS:
            oof[seed][c][va] = r[c]
        train_accs.append(r["train_acc"])
        print(f"  seed {seed} fold {k+1}/5 (train_acc {r['train_acc']:.3f}) [{time.time()-t0:.0f}s]", end="\r")
    acc = ((oof[seed]["normal"] > 0.5).astype(int) == y).mean()
    print(f"seed {seed}: OOF acc = {acc:.3f}" + " " * 30)

print(f"\n총 {time.time()-t0:.0f}초 | 평균 학습셋 정확도 {np.mean(train_accs):.3f}")

## 6. 판정 — 학습 성공 여부를 **먼저** 확인한다

1차 노트북의 결함이 여기 있었다. 학습이 됐는지 보지 않고 절제 실험 결과를
곧바로 "문맥 미사용"으로 해석했다. **학습이 실패한 모델의 절제 실험은
아무 의미가 없다** — 안 배운 모델은 history를 바꾸든 지우든 똑같이 0.5를 낸다.

그래서 v2는 게이트를 둔다.

1. 학습 성공했나? (OOF ≥ 합격선, 평균 학습셋 정확도 ≥ 0.90)
2. **통과했을 때만** 절제 실험을 해석한다.


In [ ]:
from sklearn.metrics import roc_auc_score

rows = []
for c in CONDS:
    accs = [(((oof[s][c] > 0.5).astype(int)) == y).mean() for s in SEEDS]
    aucs = [roc_auc_score(y, oof[s][c]) for s in SEEDS]
    rows.append({"조건": c, "정확도": f"{np.mean(accs):.3f} ± {np.std(accs):.3f}",
                 "AUC": f"{np.mean(aucs):.3f} ± {np.std(aucs):.3f}", "_a": np.mean(accs)})
abl = pd.DataFrame(rows)
print(abl[["조건", "정확도", "AUC"]].to_string(index=False))

a_norm, a_swap, a_empty = abl["_a"].values
drop_swap = a_norm - a_swap
mean_train = np.mean(train_accs)

print(f"\nOOF(normal) {a_norm:.3f} | 합격선 {PASS_BAR:.3f} | 학습셋 {mean_train:.3f}")
print(f"swapped 하락폭 {drop_swap:+.3f} | empty 하락폭 {a_norm - a_empty:+.3f}")
print("=" * 64)

TRAINED = (a_norm >= PASS_BAR) and (mean_train >= 0.90)

if not TRAINED:
    print("[학습 실패] 절제 실험을 해석하지 않는다.")
    print(f"  OOF {a_norm:.3f} < 합격선 {PASS_BAR:.3f} — 누출된 데이터에서 TF-IDF({BASE_RESP:.3f})도")
    print("  맞히는 패턴을 149M 인코더가 못 맞혔다. 데이터가 아니라 학습 설정 문제다.")
    print(f"  → EPOCHS {EPOCHS} → {EPOCHS*2}, LR {LR} → 8e-5 로 §3부터 재실행.")
elif a_empty > a_norm + 0.02:
    print("[이상] 문맥을 지웠을 때 성능이 더 높다. 학습이 불안정하다는 신호다.")
    print("  → seed별 편차를 확인하고 epoch을 늘려 재실행할 것.")
elif drop_swap < 0.10:
    print("[문맥 미사용] 학습은 됐으나 history를 다른 방 것으로 바꿔도 성능이 유지된다.")
    print("  → 모델이 응답 문체만 본다. 이건 데이터 문제이고, 예상된 결과다.")
    print("  → training_dataset_v3_1000.csv 로 CSV_PATH를 바꿔 재실행하면 대조군이 나온다.")
elif drop_swap < 0.30:
    print("[부분 사용] 문맥을 보긴 하지만 응답 표면 의존이 남아 있다.")
else:
    print("[문맥 사용] 문맥을 바꾸면 판정이 뒤집힌다. 의도한 대로 동작한다.")
print("=" * 64)

## 7. 성능 지표

In [ ]:
from sklearn.metrics import (classification_report, confusion_matrix,
                             precision_score, recall_score, f1_score)

p_mean = np.mean([oof[s]["normal"] for s in SEEDS], axis=0)
pred = (p_mean > 0.5).astype(int)

print(classification_report(y, pred, target_names=["적절", "부적절"], digits=3))
cm = confusion_matrix(y, pred)
print("혼동행렬        예측:적절  예측:부적절")
print(f"  실제 적절      {cm[0,0]:5d}     {cm[0,1]:5d}   <- false positive (멀쩡한 메시지에 팝업)")
print(f"  실제 부적절    {cm[1,0]:5d}     {cm[1,1]:5d}")

print(f"\n확률 분포: 평균 {p_mean.mean():.3f} | 표준편차 {p_mean.std():.3f} | "
      f"확신 비율 {(((p_mean<0.2)|(p_mean>0.8)).mean()):.0%}")

print("\n=== 기준선 대조 ===")
print(pd.DataFrame([
    {"방법": "response-only (TF-IDF)", "정확도": f"{BASE_RESP:.3f}"},
    {"방법": "history-only  (TF-IDF)", "정확도": f"{BASE_HIST:.3f}"},
    {"방법": "A.X-Encoder (empty)",    "정확도": f"{a_empty:.3f}"},
    {"방법": "A.X-Encoder (normal)",   "정확도": f"{a_norm:.3f}"},
]).to_string(index=False))

In [ ]:
print("thr   precision  recall     F1     팝업률")
best = None
for thr in np.arange(0.05, 1.00, 0.05):
    pr = (p_mean > thr).astype(int)
    if pr.sum() == 0:
        continue
    pc = precision_score(y, pr, zero_division=0)
    rc = recall_score(y, pr, zero_division=0)
    f1 = f1_score(y, pr, zero_division=0)
    print(f"{thr:.2f}    {pc:.3f}     {rc:.3f}   {f1:.3f}    {pr.mean():.3f}")
    if best is None or f1 > best[3]:
        best = (thr, pc, rc, f1)
print(f"\nF1 최대: thr={best[0]:.2f} (precision {best[1]:.3f} / recall {best[2]:.3f})")

df_out = df.copy()
df_out["prob_부적절"] = p_mean.round(3)
df_out["예측"] = np.where(pred == 1, "부적절", "적절")
df_out["정답여부"] = np.where(pred == y, "O", "X")
df_out.to_csv("oof_predictions.csv", index=False, encoding="utf-8-sig")
print(f"\n오분류 {(pred != y).sum()}행 | 저장: oof_predictions.csv")

## 8. 다음 단계

### §6이 [통과]로 나왔다면

`CSV_PATH`를 **`training_dataset_v3_1000.csv`** 로 바꾸고 전부 재실행한다.
`MAX_LEN`은 512로 올린다(긴 문맥 행이 있다).

두 실행의 `swapped 하락폭`을 나란히 놓으면 발표 슬라이드가 한 장 나온다.

| | 100행 (누출) | v3 1000행 (통제) |
|---|---|---|
| response-only | 0.940 | 0.400 |
| swapped 하락폭 | ? | ? |

**100행에서는 안 떨어지고 1000행에서는 떨어지는 그림**이 나와야 정상이고,
그게 "왜 데이터를 다시 만들어야 했는가"의 실증이 된다.

### §6이 또 [학습 실패]로 나왔다면

`EPOCHS=30`, `LR=8e-5`로 올려 §3부터 재실행한다.
그래도 학습셋 정확도가 0.9를 못 넘으면 `LR=1e-4`까지 올려본다.
100행 암기는 149M 모델에게 어려운 일이 아니므로, 반드시 넘어야 하는 선이다.

### 시간이 없다면

발표까지 남은 일정을 감안하면, 100행은 **파이프라인 점검용**으로만 쓰고
곧바로 v3 1000행으로 넘어가는 것도 합리적이다.
어차피 100행에서 나올 최선의 결과는 "누출된 데이터라 문맥을 안 배운다"이고,
그건 이미 §2의 response-only 0.940으로 증명돼 있다.
